In [8]:
import os
import re
import json
import unicodedata
from datetime import datetime

import pandas as pd

# =========================
# CẤU HÌNH ĐƯỜNG DẪN
# =========================
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("D:\\TTTN\\Project\\scripts\\preprocessing.ipynb")))
RAW_INPUT_PATH = os.path.join(BASE_DIR, "data_topcv", "topcv_all_fields_merged_2026-03-14_16-18-23.xlsx")
OUTPUT_DIR = os.path.join(BASE_DIR, "data_processed")

print("BASE_DIR         :", BASE_DIR)
print("RAW_INPUT_PATH   :", RAW_INPUT_PATH)
print("OUTPUT_DIR       :", OUTPUT_DIR)
print("File tồn tại?"   , os.path.exists(RAW_INPUT_PATH))

BASE_DIR         : D:\TTTN\Project
RAW_INPUT_PATH   : D:\TTTN\Project\data_topcv\topcv_all_fields_merged_2026-03-14_16-18-23.xlsx
OUTPUT_DIR       : D:\TTTN\Project\data_processed
File tồn tại? True


In [9]:
# Dictionary mapping skill
SKILL_DICT = {
    "python": ["python"],
    "sql": ["sql", "mysql", "postgresql", "sql server", "mssql"],
    "excel": ["excel", "microsoft excel"],
    "power_bi": ["power bi", "powerbi"],
    "tableau": ["tableau"],
    "pandas": ["pandas"],
    "numpy": ["numpy"],
    "scikit_learn": ["scikit-learn", "sklearn"],
    "machine_learning": ["machine learning", "ml"],
    "deep_learning": ["deep learning", "dl"],
    "spark": ["spark", "apache spark"],
    "hadoop": ["hadoop"],
    "airflow": ["airflow"],
    "docker": ["docker"],
    "git": ["git", "github", "gitlab"],
    "statistics": ["thống kê", "statistics", "statistical"],
    "data_visualization": ["trực quan hóa dữ liệu", "data visualization", "visualization"],
}

print("Số skill trong dict:", len(SKILL_DICT))
print("Các skill:", list(SKILL_DICT.keys()))

Số skill trong dict: 17
Các skill: ['python', 'sql', 'excel', 'power_bi', 'tableau', 'pandas', 'numpy', 'scikit_learn', 'machine_learning', 'deep_learning', 'spark', 'hadoop', 'airflow', 'docker', 'git', 'statistics', 'data_visualization']


In [10]:
# Chuẩn hóa giá trị trống
def normalize_empty_value(val):
    if pd.isna(val):
        return None
    val_str = str(val).strip()
    if not val_str or val_str.lower() in {"nan", "none", "null"}:
        return None
    return val_str

# Lấy giá trị đầu tiên không rỗng từ danh sách
def first_non_empty(*values):
    for v in values:
        v = normalize_empty_value(v)
        if v is not None:
            return v
    return None

# Chuẩn hóa unicode
def normalize_unicode(text: str) -> str:
    if text is None:
        return ""
    return unicodedata.normalize("NFC", str(text))

# Làm sạch văn bản
def clean_text(text: str) -> str:
    text = normalize_empty_value(text)
    if text is None:
        return ""
    
    text = normalize_unicode(text)
    text = text.lower()
    text = re.sub(r"<[^>]+>", " ", text)           # bỏ html
    text = re.sub(r"[\r\n\t]+", " ", text)         # xuống dòng, tab
    text = text.replace("•", " ").replace("▪", " ").replace("✅", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


print("Các hàm hỗ trợ đã định nghĩa.")

Các hàm hỗ trợ đã định nghĩa.


In [12]:
# Load dữ liệu thô
def load_raw_data(input_path: str) -> pd.DataFrame:
    ext = os.path.splitext(input_path)[1].lower()
    if ext == ".xlsx":
        df = pd.read_excel(input_path)
    elif ext == ".csv":
        df = pd.read_csv(input_path)
    else:
        raise ValueError(f"Định dạng file không hỗ trợ: {ext}")
    
    print(f"[INFO] Loaded raw data: {df.shape[0]} rows x {df.shape[1]} cols")
    return df


# Thực thi
raw_df = load_raw_data(RAW_INPUT_PATH)

# Kiểm tra nhanh
display(raw_df.head(3))
print("\nColumns:\n", raw_df.columns.tolist())

[INFO] Loaded raw data: 326 rows x 33 cols


,job_url,source_field_name,field_count,title,detail_title,company_name,company_name_full,company_url,company_url_from_job,salary_list,...,desc_mota,desc_yeucau,desc_quyenloi,company_website,company_scale_from_job,company_scale,company_field_from_job,company_address_from_job,company_address,company_description
0,https://www.topcv.vn/brand/beeogisticscorporat...,Data Analyst,1,Junior FP&A Analyst - Hồ Chí Minh,Junior FP&A Analyst - Hồ Chí Minh,Bee Logistics Corporation,Bee Logistics Corporation,https://www.topcv.vn/brand/beeogisticscorporat...,https://www.topcv.vn/brand/beeogisticscorporat...,12 - 20 triệu,...,"– Overseeing all financial planning, reporting...",– At least 2 year’ experience in fthe inance/a...,– Competitive salary according to personal exp...,http://www.beelogistics.com/,NaN,500-1000,NaN,NaN,"Addr: 11th Floor, TTC Tower, 253 Hoang Van Thu...",Xuất phát với ước mơ xây dựng một doanh nghiệp...
1,https://www.topcv.vn/brand/educa/tuyen-dung/da...,Data Analyst,1,Data Governance Specialist,Data Governance Specialist,EDUPIA,EDUPIA,https://www.topcv.vn/brand/educa?id=17270,https://www.topcv.vn/brand/educa?id=17270,20 - 30 triệu,...,"− Xây dựng, triển khai và duy trì khung Data G...","− Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên...",Mức lương thỏa thuận theo năng lực. Thử việc h...,https://edupia.vn/,NaN,500-1000,NaN,NaN,"Trụ sở: Tầng 2,3,5,6 - Tòa Vincem Comatce Towe...","Được thành lập năm 2018, Edupia là Edtech lớn ..."
2,https://www.topcv.vn/brand/educa/tuyen-dung/pr...,AI Engineer,1,Project Manager Dự Án AI HUB,Project Manager Dự Án AI HUB,EDUPIA,EDUPIA,https://www.topcv.vn/brand/educa?id=17270,https://www.topcv.vn/brand/educa?id=17270,30 - 35 triệu,...,1. Lập kế hoạch & Xây dựng chiến lược AI (20%)...,Tối thiểu 2–3 năm kinh nghiệm ở vị trí Product...,Tinh thần: - Làm việc trong môi trường start-u...,https://edupia.vn/,NaN,500-1000,NaN,NaN,"Trụ sở: Tầng 2,3,5,6 - Tòa Vincem Comatce Towe...","Được thành lập năm 2018, Edupia là Edtech lớn ..."



Columns:
 ['job_url', 'source_field_name', 'field_count', 'title', 'detail_title', 'company_name', 'company_name_full', 'company_url', 'company_url_from_job', 'salary_list', 'detail_salary', 'address_list', 'detail_location', 'exp_list', 'detail_experience', 'deadline', 'tags', 'job_level', 'education_level', 'job_quantity', 'employment_type', 'working_addresses', 'working_times', 'desc_mota', 'desc_yeucau', 'desc_quyenloi', 'company_website', 'company_scale_from_job', 'company_scale', 'company_field_from_job', 'company_address_from_job', 'company_address', 'company_description']


In [22]:
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 326 entries, 0 to 325
Data columns (total 33 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   job_url                   326 non-null    object
 1   source_field_name         326 non-null    object
 2   field_count               326 non-null    int64 
 3   title                     326 non-null    object
 4   detail_title              326 non-null    object
 5   company_name              326 non-null    object
 6   company_name_full         326 non-null    object
 7   company_url               326 non-null    object
 8   company_url_from_job      326 non-null    object
 9   salary_list               326 non-null    object
 10  detail_salary             326 non-null    object
 11  address_list              326 non-null    object
 12  detail_location           313 non-null    object
 13  exp_list                  326 non-null    object
 14  detail_experience         

In [13]:
# Gộp cột đại diện
def merge_semantic_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame()
    out["job_url"] = df.get("job_url")
    out["source_field_name"] = df.get("source_field_name")
    out["field_count"] = df.get("field_count")

    out["job_title_raw"] = [first_non_empty(a, b) for a, b in zip(df.get("detail_title"), df.get("title"))]
    
    out["salary_raw"] = [first_non_empty(a, b) for a, b in zip(df.get("detail_salary"), df.get("salary_list"))]
    out["location_raw"] = [first_non_empty(a, b) for a,b in zip(df.get("detail_location"), df.get("address_list"))]
    out["working_addresses_raw"] = df.get("working_addresses")
    out["working_times_raw"] = df.get("working_times")
    out["experience_raw"] = [first_non_empty(a, b) for a, b in zip(df.get("detail_experience"), df.get("exp_list"))]
    out["tags_raw"] = df.get("tags")
    
    out["job_level_raw"] = df.get("job_level")
    out["education_level_raw"] = df.get("education_level")
    out["employment_type_raw"] = df.get("employment_type")
    out["job_quantity"] = df.get("job_quantity")
    out["deadline_raw"] = df.get("deadline")
    
    out["description_raw"]  = df.get("desc_mota")
    out["requirements_raw"] = df.get("desc_yeucau")
    out["benefits_raw"]     = df.get("desc_quyenloi")

    out["company_name_raw"] = [first_non_empty(a, b) for a, b in zip(df.get("company_name_full"), df.get("company_name"))]
    out["company_url"] = [first_non_empty(a, b) for a, b in zip(df.get("company_url_from_job"), df.get("company_url"))]
    out["company_website_raw"] = df.get("company_website")
    out["company_scale_raw"] = [first_non_empty(a, b) for a, b in zip(df.get("company_scale_from_job"), df.get("company_scale"))]
    out["company_field_raw"] = df.get("company_field_from_job")
    out["company_address_raw"] = [first_non_empty(a, b) for a, b in zip(df.get("company_address_from_job"), df.get("company_address"))]
    out["company_description_raw"] = df.get("company_description")

    print(f"[INFO] After merging: {out.shape[0]} rows x {out.shape[1]} cols")
    return out


# Chạy
df = merge_semantic_columns(raw_df)
display(df.head(3))
df.info()

[INFO] After merging: 326 rows x 25 cols


,job_url,source_field_name,field_count,job_title_raw,salary_raw,location_raw,working_addresses_raw,working_times_raw,experience_raw,tags_raw,...,description_raw,requirements_raw,benefits_raw,company_name_raw,company_url,company_website_raw,company_scale_raw,company_field_raw,company_address_raw,company_description_raw
0,https://www.topcv.vn/brand/beeogisticscorporat...,Data Analyst,1,Junior FP&A Analyst - Hồ Chí Minh,12 - 20 triệu,Hồ Chí Minh,(đã được cập nhật theo Danh mục Hành chính mới...,Thứ 2 - Thứ 6 (từ 08:00 đến 17:30) Thứ 7 (từ 0...,2 năm,Chuyên môn Data Analyst; Tài chính; Kế toán,...,"– Overseeing all financial planning, reporting...",– At least 2 year’ experience in fthe inance/a...,– Competitive salary according to personal exp...,Bee Logistics Corporation,https://www.topcv.vn/brand/beeogisticscorporat...,http://www.beelogistics.com/,500-1000,NaN,"Addr: 11th Floor, TTC Tower, 253 Hoang Van Thu...",Xuất phát với ước mơ xây dựng một doanh nghiệp...
1,https://www.topcv.vn/brand/educa/tuyen-dung/da...,Data Analyst,1,Data Governance Specialist,20 - 30 triệu,Hà Nội,(đã được cập nhật theo Danh mục Hành chính mới...,Thứ 2 - Thứ 6 (từ 08:00 đến 17:30) Thứ 7 (từ 0...,2 năm,Chuyên môn Data Analyst,...,"− Xây dựng, triển khai và duy trì khung Data G...","− Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên...",Mức lương thỏa thuận theo năng lực. Thử việc h...,EDUPIA,https://www.topcv.vn/brand/educa?id=17270,https://edupia.vn/,500-1000,NaN,"Trụ sở: Tầng 2,3,5,6 - Tòa Vincem Comatce Towe...","Được thành lập năm 2018, Edupia là Edtech lớn ..."
2,https://www.topcv.vn/brand/educa/tuyen-dung/pr...,AI Engineer,1,Project Manager Dự Án AI HUB,30 - 35 triệu,Hà Nội,(đã được cập nhật theo Danh mục Hành chính mới...,Thứ 2 - Thứ 6 (từ 08:00 đến 17:30) Thứ 7 (từ 0...,2 năm,Chuyên môn IT Project Manager,...,1. Lập kế hoạch & Xây dựng chiến lược AI (20%)...,Tối thiểu 2–3 năm kinh nghiệm ở vị trí Product...,Tinh thần: - Làm việc trong môi trường start-u...,EDUPIA,https://www.topcv.vn/brand/educa?id=17270,https://edupia.vn/,500-1000,NaN,"Trụ sở: Tầng 2,3,5,6 - Tòa Vincem Comatce Towe...","Được thành lập năm 2018, Edupia là Edtech lớn ..."


<class 'pandas.DataFrame'>
RangeIndex: 326 entries, 0 to 325
Data columns (total 25 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   job_url                  326 non-null    str  
 1   source_field_name        326 non-null    str  
 2   field_count              326 non-null    int64
 3   job_title_raw            326 non-null    str  
 4   salary_raw               326 non-null    str  
 5   location_raw             326 non-null    str  
 6   working_addresses_raw    326 non-null    str  
 7   working_times_raw        296 non-null    str  
 8   experience_raw           326 non-null    str  
 9   tags_raw                 326 non-null    str  
 10  job_level_raw            326 non-null    str  
 11  education_level_raw      326 non-null    str  
 12  employment_type_raw      326 non-null    str  
 13  job_quantity             326 non-null    str  
 14  deadline_raw             326 non-null    str  
 15  description_raw  

In [ ]:
df[["job_url", "company_address_from_job", "company_address"]].head(10)

In [31]:
df["job_title_clean"]     = df["job_title_raw"].apply(clean_text)
df["description_clean"]   = df["description_raw"].apply(clean_text)
df["requirements_clean"]  = df["requirements_raw"].apply(clean_text)
df["benefits_clean"]      = df["benefits_raw"].apply(clean_text)

# Kiểm tra mẫu
print("Sample sau khi clean text:")
display(df[["job_title_raw", "job_title_clean", 
            "requirements_raw", "requirements_clean"]].head(8))

Sample sau khi clean text:


,job_title_raw,job_title_clean,requirements_raw,requirements_clean
0,Junior FP&A Analyst - Hồ Chí Minh,junior fp&a analyst - hồ chí minh,– At least 2 year’ experience in fthe inance/a...,– at least 2 year’ experience in fthe inance/a...
1,Data Governance Specialist,data governance specialist,"− Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên...","− đã tốt nghiệp đh trở lên, ưu tiên các chuyên..."
2,Project Manager Dự Án AI HUB,project manager dự án ai hub,Tối thiểu 2–3 năm kinh nghiệm ở vị trí Product...,tối thiểu 2–3 năm kinh nghiệm ở vị trí product...
3,AI Engineer,ai engineer,Kinh nghiệm AI/ML ứng dụng thực tế Python và x...,kinh nghiệm ai/ml ứng dụng thực tế python và x...
4,AI Engineer,ai engineer,"Tốt nghiệp Đại học chuyên ngành CNTT, Khoa học...","tốt nghiệp đại học chuyên ngành cntt, khoa học..."
5,Data Analyst,data analyst,Trên 3 năm kinh nghiệm ở vị trí Data Analyst C...,trên 3 năm kinh nghiệm ở vị trí data analyst c...
6,Data Engineer,data engineer,- Tốt nghiệp Đại học trở lên các chuyên ngành ...,- tốt nghiệp đại học trở lên các chuyên ngành ...
7,Fresher Data Engineer,fresher data engineer,· Là sinh viên năm cuối/đã tốt nghiệp Đại học ...,· là sinh viên năm cuối/đã tốt nghiệp đại học ...


- tiếp tục từ đây

In [9]:
def normalize_location(text: str) -> str:
    text = clean_text(text)
    if not text:
        return ""
    mapping = {
        "tp.hcm": "ho chi minh", "tphcm": "ho chi minh", "tp hcm": "ho chi minh",
        "hcm": "ho chi minh", "hồ chí minh": "ho chi minh", "sài gòn": "ho chi minh",
        "hn": "ha noi", "hà nội": "ha noi", "đà nẵng": "da nang",
    }
    for k, v in mapping.items():
        if k in text:
            return v
    return text


def parse_salary(text: str):
    text = clean_text(text)
    if not text:
        return None, None, "unknown"
    if any(x in text for x in ["thỏa thuận", "thuong luong", "cạnh tranh"]):
        return None, None, "negotiable"
    
    nums = re.findall(r"\d+(?:[.,]\d+)?", text)
    nums = [float(n.replace(",", ".")) for n in nums]
    
    if "triệu" in text or "trieu" in text:
        nums = [n * 1_000_000 for n in nums]
    elif "usd" in text:
        pass  # giữ nguyên
    
    nums = [int(round(n)) for n in nums]
    
    if len(nums) >= 2:
        return nums[0], nums[1], "range"
    if len(nums) == 1:
        return nums[0], nums[0], "fixed"
    return None, None, "unknown"


def parse_experience(text: str):
    text = clean_text(text)
    if not text:
        return None, None, "unknown"
    if "không yêu cầu" in text:
        return 0, 0, "no_requirement"
    if any(x in text for x in ["thực tập", "intern", "fresher"]):
        return 0, 1, "intern_fresher"
    
    nums = [int(x) for x in re.findall(r"\d+", text)]
    if len(nums) >= 2:
        return nums[0], nums[1], "range"
    if len(nums) == 1:
        if any(x in text for x in ["+", "trên", "ít nhất"]):
            return nums[0], None, "min_only"
        return nums[0], nums[0], "fixed"
    return None, None, "unknown"


# Áp dụng
df["location_normalized"] = df["location_raw"].apply(normalize_location)

salary_parsed = df["salary_raw"].apply(parse_salary)
df["salary_min"]  = [x[0] for x in salary_parsed]
df["salary_max"]  = [x[1] for x in salary_parsed]
df["salary_type"] = [x[2] for x in salary_parsed]

exp_parsed = df["experience_raw"].apply(parse_experience)
df["experience_min_years"] = [x[0] for x in exp_parsed]
df["experience_max_years"] = [x[1] for x in exp_parsed]
df["experience_type"]      = [x[2] for x in exp_parsed]


# Kiểm tra
print("Location value counts:\n", df["location_normalized"].value_counts().head(10))
print("\nSalary types:\n", df["salary_type"].value_counts())
print("\nExperience types:\n", df["experience_type"].value_counts())

Location value counts:
 location_normalized
ha noi            235
ho chi minh        78
da nang             7
hưng yên            3
cần thơ             2
bắc ninh (mới)      1
nghệ an             1
vĩnh long           1
hải phòng           1
Name: count, dtype: int64

Salary types:
 salary_type
unknown    162
range      140
fixed       27
Name: count, dtype: int64

Experience types:
 experience_type
fixed             262
no_requirement     54
min_only           13
Name: count, dtype: int64


In [10]:
def extract_skills(text: str, skill_dict: dict) -> list:
    text = clean_text(text)
    found = set()
    for canonical, aliases in skill_dict.items():
        for alias in aliases:
            alias_clean = clean_text(alias)
            pattern = rf"(?<!\w){re.escape(alias_clean)}(?!\w)"
            if re.search(pattern, text, re.IGNORECASE):
                found.add(canonical)
                break
    return sorted(found)


# Nguồn text để extract skill
skill_source = (
    df["job_title_clean"].fillna("") + " " +
    df["requirements_clean"].fillna("") + " " +
    df["description_clean"].fillna("")
)

df["skills_normalized"] = skill_source.apply(lambda x: extract_skills(x, SKILL_DICT))
df["skills_normalized_str"] = df["skills_normalized"].apply(lambda x: " | ".join(x) if x else "")

# Kiểm tra
print("Top 15 skill phổ biến:")
skill_flat = [s for sublist in df["skills_normalized"] for s in sublist]
pd.Series(skill_flat).value_counts().head(15)

Top 15 skill phổ biến:


python                147
sql                   144
machine_learning       96
power_bi               62
statistics             53
excel                  50
tableau                50
spark                  45
docker                 45
data_visualization     43
deep_learning          38
git                    35
airflow                30
scikit_learn           24
hadoop                 22
Name: count, dtype: int64

In [11]:
def build_job_text(row):
    parts = []
    if row.get("job_title_clean"):
        parts.append(f"[title] {row['job_title_clean']}")
    if row.get("requirements_clean"):
        parts.append(f"[requirements] {row['requirements_clean']}")
    if row.get("description_clean"):
        parts.append(f"[description] {row['description_clean']}")
    if row.get("skills_normalized"):
        parts.append(f"[skills] {' | '.join(row['skills_normalized'])}")
    return " ".join(parts).strip()


df["job_text"] = df.apply(build_job_text, axis=1)

# Kiểm tra mẫu
print("Ví dụ job_text:")
print(df["job_text"].iloc[0][:600] + "..." if len(df["job_text"].iloc[0]) > 600 else df["job_text"].iloc[0])

Ví dụ job_text:
[title] junior fp&a analyst - hồ chí minh [requirements] – at least 2 year’ experience in fthe inance/accounting area within big/complex organizations and/or audit services in reputable finance consulting firms. – bachelor's degree in finance/accounting, similar and above. – holding professional qualifications like acca, cpa, icaew... is an advantage. – experience in the automotive/vehicle/oem manufacturing business environment is an advantage. – interpretation and analysis skills, diligence, and honesty in communication. – ability to manage stress and work in a professional environment, inclu...


In [12]:
def select_final_columns(df):
    final_cols = [
        "job_url", "source_field_name", "field_count",
        "job_title_raw", "job_title_clean",
        "company_name_raw", "company_url",
        "salary_raw", "salary_min", "salary_max", "salary_type",
        "location_raw", "location_normalized",
        "experience_raw", "experience_min_years", "experience_max_years", "experience_type",
        "job_level_raw", "education_level_raw", "employment_type_raw", "deadline_raw",
        "company_scale_raw", "company_address_raw", "company_description_raw",
        "description_raw", "description_clean",
        "requirements_raw", "requirements_clean",
        "benefits_raw", "benefits_clean",
        "skills_normalized", "skills_normalized_str",
        "job_text"
    ]
    existing = [c for c in final_cols if c in df.columns]
    return df[existing].copy()


final_df = select_final_columns(df)
print("Final shape:", final_df.shape)
display(final_df.head(3))

Final shape: (329, 33)


,job_url,source_field_name,field_count,job_title_raw,job_title_clean,company_name_raw,company_url,salary_raw,salary_min,salary_max,...,company_description_raw,description_raw,description_clean,requirements_raw,requirements_clean,benefits_raw,benefits_clean,skills_normalized,skills_normalized_str,job_text
0,https://www.topcv.vn/brand/beeogisticscorporat...,Data Analyst,1,Junior FP&A Analyst - Hồ Chí Minh,junior fp&a analyst - hồ chí minh,Bee Logistics Corporation,https://www.topcv.vn/brand/beeogisticscorporat...,12 - 20 triệu,12000000.0,20000000.0,...,Xuất phát với ước mơ xây dựng một doanh nghiệp...,"– Overseeing all financial planning, reporting...","– overseeing all financial planning, reporting...",– At least 2 year’ experience in fthe inance/a...,– at least 2 year’ experience in fthe inance/a...,– Competitive salary according to personal exp...,– competitive salary according to personal exp...,[],,[title] junior fp&a analyst - hồ chí minh [req...
1,https://www.topcv.vn/brand/chinhanhtranscosmos...,Data Analyst | Data Labeling (Gán nhãn dữ liệu),2,Data Analyst Workforce Management,data analyst workforce management,CÔNG TY TNHH TRANSCOSMOS VIỆT NAM,https://www.topcv.vn/brand/chinhanhtranscosmos...,Thoả thuận,NaN,NaN,...,"Được thành lập từ năm 1966 tại Shibuya, Tokyo,...",NaN,,NaN,,NaN,,[],,[title] data analyst workforce management
2,https://www.topcv.vn/brand/educa/tuyen-dung/da...,Data Analyst,1,Data Governance Specialist,data governance specialist,EDUPIA,https://www.topcv.vn/brand/educa?id=17270,20 - 30 triệu,20000000.0,30000000.0,...,"Được thành lập năm 2018, Edupia là Edtech lớn ...","− Xây dựng, triển khai và duy trì khung Data G...","− xây dựng, triển khai và duy trì khung data g...","− Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên...","− đã tốt nghiệp đh trở lên, ưu tiên các chuyên...",Mức lương thỏa thuận theo năng lực. Thử việc h...,mức lương thỏa thuận theo năng lực. thử việc h...,[sql],sql,[title] data governance specialist [requiremen...


In [13]:
def save_processed_data(df, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    ts = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    
    csv_path = os.path.join(output_dir, f"jobs_nlp_ready_{ts}.csv")
    xlsx_path = os.path.join(output_dir, f"jobs_nlp_ready_{ts}.xlsx")
    
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"Saved CSV → {csv_path}")
    
    try:
        df.to_excel(xlsx_path, index=False)
        print(f"Saved XLSX → {xlsx_path}")
    except Exception as e:
        print(f"Không lưu được Excel: {e}")


# Lưu
save_processed_data(final_df, OUTPUT_DIR)

Saved CSV → D:\Thực tập tốt nghiệp\Project\data_processed\jobs_nlp_ready_2026-03-14_18-54-36.csv
Saved XLSX → D:\Thực tập tốt nghiệp\Project\data_processed\jobs_nlp_ready_2026-03-14_18-54-36.xlsx


In [14]:
# Export nhanh để test (không timestamp)
quick_path = os.path.join(OUTPUT_DIR, "jobs_nlp_ready_latest_debug.csv")
final_df.to_csv(quick_path, index=False, encoding="utf-8-sig")
print("Quick debug file:", quick_path)

Quick debug file: D:\Thực tập tốt nghiệp\Project\data_processed\jobs_nlp_ready_latest_debug.csv
